# NB04 — Prioritized Gap List (Top-100 per biome)

**Environment:** Local Python (with optional Spark step for enrichment fetching)

**Purpose:** For each biome with ≥ 100 genomes, produce a ranked list of gene clusters that are (a) core within that biome, (b) have no PDB structural evidence, (c) have AF present (confident or low-confidence), (d) have functional annotation suggesting biome relevance. Annotate each with predicted domain(s), MSA depth, species range. Cross-reference against `alphafold_msa_annotation` paradox proteins.

**Inputs:**
- `s3a://.../cluster_biome_coverage.parquet` (from NB02) — or a locally-materialized subset
- `../../../alphafold_msa_annotation/data/paradox_top1000.csv` — for cross-reference
- InterProScan domain data (per-cluster, top predicted family) — fetched via Spark or from cached export

**Output:**
- `data/priority_gap_list.csv` — top 100 per biome-compartment (≤ 1000 rows total)
- `data/priority_gap_list_annotated.csv` — with per-candidate Pfam, TM helix count, species range

In [ ]:
import pandas as pd
import numpy as np

## Load per-cluster-biome coverage (materialize a manageable subset locally first)

In [ ]:
# Assumes NB02 wrote a filtered CSV subset for local processing (cluster-level, only rows meeting gap criteria).
# If not, run the Spark-side filter in NB02 to write a small CSV.
cbc = pd.read_csv("data/gap_candidates_prefilter.csv")  # produced from cluster_biome_coverage.parquet
print(cbc.shape)

## Score & rank per biome

In [ ]:
# Filter: core in biome, no PDB, AF present
cand = cbc[
    (cbc["is_core"] == True)
    & (cbc["pdb_tier"] == "none")
    & (cbc["af_tier"].isin(["confident", "low_confidence"]))
].copy()

# Score: biome prevalence weight * AF-low bonus * has-annotation flag
cand["has_annotation"] = (
    cand["kegg_orthology_id"].notna()
    | cand["ec"].notna()
    | ~cand["product"].fillna("hypothetical").str.contains("hypothetical", case=False)
).astype(int)

cand["score"] = (
    cand.groupby(["compartment", "gene_cluster_id"])["n_genomes"].transform("first").fillna(1)
    * (1 + (cand["af_tier"] == "low_confidence").astype(int))
    * (0.2 + cand["has_annotation"])
)

priority = (cand.sort_values(["compartment", "score"], ascending=[True, False])
                .groupby("compartment").head(100))
priority.to_csv("data/priority_gap_list.csv", index=False)
print(f"Priority candidates: {len(priority):,} across {priority['compartment'].nunique()} compartments")

## Cross-reference against paradox protein list (`alphafold_msa_annotation`)

In [ ]:
paradox = pd.read_csv("../../../alphafold_msa_annotation/data/paradox_top1000.csv")
overlap = priority.merge(
    paradox[["gene_cluster_id"]].drop_duplicates(),
    on="gene_cluster_id", how="inner"
)
print(f"Overlap with top-1000 paradox proteins: {len(overlap):,} / {len(priority):,} "
      f"({100 * len(overlap) / max(len(priority), 1):.1f}%)")

## Per-candidate annotation (Pfam, TM helix, species range)

For each of the top-100 per biome, join to InterProScan for Pfam family, and to a TMHMM/DeepTMHMM prediction (if cached in BERDL) for transmembrane count. Also produce species-range size from `marker_gene_clusters`.

In [ ]:
# Placeholder — implement once IPS subset is materialized locally, or run this as a small Spark job.
# ips = pd.read_csv("data/ips_for_priority.csv")
# annotated = priority.merge(ips, on="gene_cluster_id", how="left")
# annotated.to_csv("data/priority_gap_list_annotated.csv", index=False)